**Preliminari**

In [75]:
import sys
import os
from dotenv import load_dotenv  
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import json
from utils.schema_json import ReportData, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset, DatasetDict

Added to sys.path: C:\Users\lucat\PythonRepositories\PRIN


**Impostiamo il device, scheda video se disponibile**

In [76]:
print(f'{torch.cuda.is_available() = }')  # True se la GPU è disponibile
print(f'{torch.cuda.device_count() = }')  # Numero di GPU disponibili
print(f'{torch.cuda.get_device_name(0) = }')  # Nome della GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device = }')

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
torch.cuda.get_device_name(0) = 'NVIDIA GeForce GTX 1060 6GB'
device = device(type='cuda')


**Huggingface login**

In [77]:
# Set the API key for HuggingFace
load_dotenv()  # Load environment variables from .env file
hf_api_key = os.getenv("HF_TOKEN")
login(token=hf_api_key)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


**Parametri**

In [78]:
# Parameters
TRAIN_FILE_NAME = "data_finetune_guido_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_guido_openai_val.jsonl"
#FILE_NAMES = (TRAIN_FILE_NAME, VALIDATION_FILE_NAME)
TIPO = 'openai'

CHECKPOINT = "bert-base-multilingual-cased"

# Fields we don't want to predict
EXCLUDED_FIELDS = (
    'ore_inizio',
    'ore_fine',
    'spessore_parietale',
    'estensione_cranio_caudale',
    'distanza_oai',
    'linfonodi_sospetti',
    'numero_depositi'
)

**Load data**

In [79]:
# Carichiamo i nostri file JSON
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
}

paths = {
    split: Path('../data/ft-dataset/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data_list = [json.loads(line) for line in f]
        data[split] = data_list

train_data, validation_data = data['train'], data['validation']

print(f"{len(train_data) = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][1]['content']) = }")  # Report text
print(f"{type(train_data[0]['messages'][2]['content']) = }")  # Annotations

len(train_data) = 116
type(train_data[0]) = <class 'dict'>
type(train_data[0]['messages'][1]['content']) = <class 'str'>
type(train_data[0]['messages'][2]['content']) = <class 'str'>


In [80]:
annotated_reports: dict[str, list[AnnotatedReport]] = {split: [] for split in file_names.keys()}
for split in annotated_reports:
    for record in data[split]:
        report_text = record['messages'][1]['content'].lower()  # Tutte lettere minuscole
        if TIPO == 'openai':
            report_data = ReportData.model_validate_json(record['messages'][2]['content'])
        else:
            report_data = ReportData.model_validate(record['messages'][2]['content'])
        annotated_reports[split].append(AnnotatedReport(report_text=report_text, report_data=report_data))

**Load model and tokenizer**

In [81]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModel.from_pretrained(CHECKPOINT)

In [82]:
# Check the maximum number of tokens for each report
max_n_tokens_train = 0
del_train = []
for i, report in enumerate(annotated_reports['train']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_train = max(max_n_tokens_train, x)
    if x > model.config.max_position_embeddings:
        del_train.append(i)
print(del_train)
print(f'{max_n_tokens_train = }')

# Check the maximum number of tokens for each report
max_n_tokens_validation = 0
del_val = []
for i, report in enumerate(annotated_reports['validation']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_validation = max(max_n_tokens_validation, x)
    if x > model.config.max_position_embeddings:
        del_val.append(i)
print(del_val)
print(f'{max_n_tokens_validation = }')

# Delete long reports
for i in del_train[::-1]:
    annotated_reports['train'].pop(i)
for i in del_val[::-1]:
    annotated_reports['validation'].pop(i)
print('deleted')

Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


[86, 95, 97, 111]
max_n_tokens_train = 659
[1, 5, 17, 26]
max_n_tokens_validation = 949
deleted


In [83]:
def create_hugging_face_dataset(annotated_reports: list[AnnotatedReport]) -> Dataset:
    text = []
    for report in annotated_reports:
        text.append(report.report_text)
    return Dataset.from_dict({'text': text})

In [84]:
dataset = DatasetDict({
    'train': create_hugging_face_dataset(annotated_reports['train']),
    'validation': create_hugging_face_dataset(annotated_reports['validation'])
})

In [85]:
def tokenize_function(examples, padding="max_length", max_length=model.config.max_position_embeddings):
    return tokenizer(examples['text'])    

In [86]:
dataset = dataset.map(tokenize_function, batched=True)
dataset.set_format('torch')
print(dataset)

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24
    })
})


In [87]:
print(dataset['train'][0]['text'])
print(dataset['train'][0]['input_ids'])
print(dataset['train'][0]['attention_mask'])

si segnala la presenza in corrispondenza del retto basso , dalla giunzione rettoanale con estensione craniale per circa 6 cm , di una formazione moderatamente iperintensa nelle sequenze t2 dipendenti che interessa circonferenzialmente il retto basso ed invia digitazioni nel mesoretto una delle quali raggiunge, a sinistra, il muscolo elavatore dell'ano di sinistra. lungo i vasi emorroidari superiori sono presenti alcuni linfonodi i maggiori del diametro di circa 1 cm. linfonodi di dimensioni inferiori ad 1 cm in sede inguinale bilaterale. esiti di pregresso intervento di turp. conclusioni:stadiazione rm t4n1
tensor([   101,  10294, 106687,  10330,  10109,  21857,  10106, 109822,  10127,
         49144,  10133,  21365,    117,  11353,  38356,  11107,  11115,  49144,
         28337,  12223,  10173, 102190,    171,  90205,  10284,  10178,  13355,
           127,  11207,    117,  10120,  10153,  29210,  18417,  18017,  10611,
           177,  69692,  27321,  10466,  13063,  10126,  78536,  

In [89]:
print(dataset['train'][0].keys())

dict_keys(['text', 'input_ids', 'token_type_ids', 'attention_mask'])


In [74]:
print(dataset['train']['input_ids'][0].shape)

torch.Size([159])


In [93]:
print(dataset['train'][0]['text'])
print(dataset['train'][0]['input_ids'])
print(dataset['train'][0]['attention_mask'])

si segnala la presenza in corrispondenza del retto basso , dalla giunzione rettoanale con estensione craniale per circa 6 cm , di una formazione moderatamente iperintensa nelle sequenze t2 dipendenti che interessa circonferenzialmente il retto basso ed invia digitazioni nel mesoretto una delle quali raggiunge, a sinistra, il muscolo elavatore dell'ano di sinistra. lungo i vasi emorroidari superiori sono presenti alcuni linfonodi i maggiori del diametro di circa 1 cm. linfonodi di dimensioni inferiori ad 1 cm in sede inguinale bilaterale. esiti di pregresso intervento di turp. conclusioni:stadiazione rm t4n1
[   101  10294 106687  10330  10109  21857  10106 109822  10127  49144
  10133  21365    117  11353  38356  11107  11115  49144  28337  12223
  10173 102190    171  90205  10284  10178  13355    127  11207    117
  10120  10153  29210  18417  18017  10611    177  69692  27321  10466
  13063  10126  78536  10112    188  10729  10120  67475  12752  10262
  82927  10466  11322  40012  